# Backpropagation from Scratch (XOR)

A 2-3-1 neural network trained **without any deep-learning library** to solve XOR.

XOR is not linearly separable, so a single neuron cannot solve it - the hidden layer is essential.

**Each training iteration:**
1. *Forward pass* - compute the prediction
2. *Backward pass* - propagate the error back through the layers (chain rule)
3. *Update* - move weights and biases a small step to reduce the error

The last cell plots the learned decision boundary.

In [3]:
# NumPy for the maths, Matplotlib for the decision-boundary plot.
import numpy as np 
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Sigmoid and its derivative.
# NOTE: sigmoid_derivative expects the already-activated output a = sigmoid(z), because sigmoid'(z) = a * (1 - a).
def sigmoid(x):
    return 1/(1+np.exp(-x))
def sigmoid_derivative(x):
    return x*(1-x)

# Dummy data
# XOR inputs
x=np.array([[0,0],[0,1],[1,0],[1,1]])
# XOR targets: 1 only when exactly one input is 1
y=np.array([[0],[1],[1],[0]])


In [ ]:
# Initialise the network and train it with backpropagation.
#initialze wights and biases
# architecture: 2 inputs -> 3 hidden neurons -> 1 output
input_neurons =2
hidden_neurons=3
output_neurons=1

# random initial weights and biases (uniform in [0, 1))
weights_hidden = np.random.uniform(size=(input_neurons,hidden_neurons))
bias_hidden=np.random.uniform(size=(1,hidden_neurons))
weights_output=np.random.uniform(size=(hidden_neurons,output_neurons))
bias_output=np.random.uniform(size=(1,output_neurons))

#hyperparameters
learning_rate=0.1
epochs=10000

#training loop
for epoch in range(epochs):
    #forward propagation
    hidden_layer_input = np.dot(x,weights_hidden)+bias_hidden
    hidden_layer_output=sigmoid(hidden_layer_input)

    output_layer_input=np.dot(hidden_layer_output,weights_output)+bias_output
    output_layer_output=sigmoid(output_layer_input)

    #Backprogation
    # output error: target - prediction
    error=y-output_layer_output
    # output-layer gradient: error * sigmoid'(output)
    output_delta=error*sigmoid_derivative(output_layer_output)

    # propagate the error back through the output weights
    hidden_error=output_delta.dot(weights_output.T)
    # hidden-layer gradient
    hidden_delta=hidden_error*sigmoid_derivative(hidden_layer_output)


    #update weights and biases
    # gradient-descent updates (+= because error is defined as y - y_hat)
    weights_output +=hidden_layer_output.T.dot(output_delta)*learning_rate
    bias_output+=np.sum(output_delta, axis=0,keepdims=True)*learning_rate
    weights_hidden+=x.T.dot(hidden_delta)*learning_rate
    bias_hidden +=np.sum(hidden_delta,axis=0,keepdims=True)*learning_rate

#print predictions after training
hidden_layer_input=np.dot(x,weights_hidden)+bias_hidden
hidden_layer_output=sigmoid(hidden_layer_input)
output_layer_input=np.dot(hidden_layer_output,weights_output)+bias_output
output_layer_output=sigmoid(output_layer_input)
print("predictions:")
print(np.round(output_layer_output))



predictions:
[[0.]
 [1.]
 [1.]
 [0.]]


In [ ]:
# Evaluate the trained network on a dense grid covering the input space and colour each region by predicted class. The black line is the 0.5 decision boundary.
#plotting the decision boundary with colored regions
x_min,x_max=x[:,0].min()-0.1,x[:,0].max()+0.1
y_min,y_max=x[:,1].min()-0.1,x[:,1].max()+0.1
xx, yy=np.meshgrid(np.arange(x_min,x_max,0.01),
                   np.arange(y_min,y_max,0.01))
z=sigmoid(np.dot(sigmoid(np.dot(np.c_[xx.ravel(),yy.ravel()],weights_hidden)+bias_hidden),weights_output)+bias_output)
z=z.reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx,yy,z,levels=[0,0.5,1],colors=['lightblue','lightcoral'],alpha=0.8)
plt.contour(xx,yy,z,levels=[0.5],colors='black')
plt.scatter(x[:,0],x[:,1],c=y.ravel(),cmap=plt.cm.RdBu, edgecolors='k')
plt.xlabel('Feature1')
plt.ylabel('Feature2')
plt.title('Neural network decidion boundary (colored regions)')
plt.show()